<a href="https://colab.research.google.com/github/rafaelrdealmeida/fundamentos_2026_lantri01/blob/main/notebooks/encontro_2026_lantri_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introdução ao Pensamento computacional

## Ferramentas/Serviços utilizados

- Google Colab (notebook com codigo)
- Github (rede social de código)
- Git (programa de versionamento de código)


## Pilares
- Decomposição
- Abstração
- Identificação de padrões
- Lógica


## Noções Gerais

- Exemplos a partir de coleta de dados
  - Site MRE - Notas de imprensa
    - [x] Link do site: https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa
    - [x] Entender a estrutura da fonte de informação
    - [ ] Realizar a coleta
    - [ ] Inserir as informações em um banco de dados
    - [ ] Utilizar as informações (analise de dados)


### Padrão de paginação das notas de imprensa
  - https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa?b_start:int=0
   - https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa?b_start:int=30
   - Pagina final (10/02/2026): 6210



## Importação de bibliotecas/pogramas utilizados neste arquivo

In [1]:
!pip install tinydb

In [2]:
# programas/bibliotecas utilizados no script/codigo
import httpx # Responsável pelas requisições web
from bs4 import BeautifulSoup # Responsável por realizar o web scraping (coletar os dados)
from tinydb import TinyDB, Query

## Criação do banco json

In [3]:
def inserir_no_banco(dados, link_noticia):
  arquivo_banco_dados = "nota_mre.json"
  db = TinyDB(arquivo_banco_dados)


  # Evitar dados repetidos no banco
  Buscar = Query()
  verificar_link = db.contains(Buscar.link == link_noticia)

  if not verificar_link:
    print("Inserindo nova informação no banco")
    db.insert(dados)
  else:
    print("Link já existe no banco. Esta informação não será inserida novamente")

In [5]:
# Variável e tipos de dados (string, lista, numero)
paginas = ["https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa?b_start:int=0"]

def acessa_pagina (link):
  print (f"Estamos na pagina:{link}")

  # Define headers para a requisição, simulando um navegador
  headers = {
      'User-Agent': "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36",
      'Accept-Language': 'en-US,en;q=0.9',
      'Accept-Encoding': 'gzip, deflate, br',
      'Connection': 'keep-alive',
  }

  timeout = httpx.Timeout(connect=20.0, read=30.0, write=20.0, pool=10.0)
  pag_web = httpx.get(link, headers=headers, timeout=timeout)
  bs = BeautifulSoup(pag_web, "html.parser")
  return bs

# loop for
# beautifulsoap >> find e find_all

for pagina in paginas:
  pagina_inteira = acessa_pagina(pagina)
  lista_noticias = pagina_inteira.find("div", attrs={"id": "content-core"}).find_all("article")
  for noticia in lista_noticias:
    # titulo
    try:
      titulo = noticia.find("h2", attrs={"class": "tileHeadline"}).text.strip()
      print(titulo)
    except:
        titulo = ""

    #link
    try:
      link_noticia = noticia.a["href"]
      print(link_noticia)
    except:
      link_noticia = ""
    # numero da nota - exemplo: NOTA À IMPRENSA Nº 72
    # numero da nota - exemplo: NOTA À IMPRENSA N° 590
    num_nota = noticia.find("span", attrs={"class": "subtitle"}).text.strip()
    # print(num_nota)
    # num_nota = noticia.find(attrs={"class": "subtitle"}).text.strip()
    num_nota = num_nota.replace("NOTA À IMPRENSA N°", "").replace("NOTA À IMPRENSA Nº", "").strip()
    print(num_nota)
    print("###")
    # data
    # horário
    data_hora = noticia.find_all("span",attrs={"class": "summary-view-icon"})
    data= data_hora[0].text.strip()
    hora = data_hora[1].text.strip()
    print(data)
    print(hora)
    conteudo = acessa_pagina (link_noticia)
    paragrafos = conteudo.find("div", attrs={"property":"rnews:articleBody"}).find_all("p")
    lista_paragrafos = []
    for paragrafo in paragrafos:
      lista_paragrafos.append(paragrafo.text.strip())
    print(lista_paragrafos)
    # função para inserir dados coletados no banco
    dados = {
        "titulo": titulo,
        "link": link_noticia,
        "data": data,
        "hora": hora,
        "num_nota": num_nota,
        "paragrafo": lista_paragrafos
    }
    inserir_no_banco(dados,link_noticia)






Estamos na pagina:https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa?b_start:int=0
Aberturas de mercado para o Brasil no Togo - Nota Conjunta MRE/MAPA
https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/aberturas-de-mercado-para-o-brasil-no-togo-nota-conjunta-mre-mapa
143
###
22/04/2026
19h20
Estamos na pagina:https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/aberturas-de-mercado-para-o-brasil-no-togo-nota-conjunta-mre-mapa
['O governo brasileiro concluiu negociações que permitirão a exportação de material genético bovino (sêmen e embriões) para o Togo.As aberturas criam oportunidades para produtores brasileiros, bem como para a prestação de serviços de consultoria e assistência técnica. Em 2025, o Brasil exportou mais de US$ 148 milhões em produtos agropecuários para o Togo, com destaque para produtos do complexo sucroalcooleiro, carnes e couro.Com esse anúncio, o agronegócio brasileiro alcança 594 a

# Transformar banco json e dataframe

- pre-analise - entendimento geral sobre o dataframe

In [9]:
import pandas as pd
import json

## Abrindo o rquivo json
with open("nota_mre.json") as f:
  raw = json.load(f)

df = pd.DataFrame.from_dict(raw["_default"], orient="index")

df


,titulo,link,data,hora,num_nota,paragrafo
1,Aberturas de mercado para o Brasil no Togo - N...,https://www.gov.br/mre/pt-br/canais_atendiment...,22/04/2026,19h20,143,[O governo brasileiro concluiu negociações que...
2,Atos adotados por ocasião da Visita de Estado ...,https://www.gov.br/mre/pt-br/canais_atendiment...,20/04/2026,13h13,142,[Foram adotados os seguintes atos por ocasião ...
3,Declaração Conjunta Brasil-Alemanha sobre apoi...,https://www.gov.br/mre/pt-br/canais_atendiment...,20/04/2026,11h12,141,[No contexto da reunião bilateral entre o Pres...
4,III Consultas Intergovernamentais de Alto Níve...,https://www.gov.br/mre/pt-br/canais_atendiment...,20/04/2026,10h26,140,[1. Sob a presidência do Chanceler Federal Fri...
5,4ª Reunião de Alto Nível do Fórum Democracia S...,https://www.gov.br/mre/pt-br/canais_atendiment...,19/04/2026,07h58,139,"[Nosotras y nosotros, los Jefes de Estado y de..."
6,Declaração Conjunta Sobre a Situação em Cuba,https://www.gov.br/mre/pt-br/canais_atendiment...,18/04/2026,16h44,138,[À luz da evolução da situação em Cuba e das c...
7,Visita Oficial do Senhor Presidente da Repúbli...,https://www.gov.br/mre/pt-br/canais_atendiment...,18/04/2026,11h33,137,[O senhor Presidente da República realizará vi...
8,Cessar-fogo no Líbano,https://www.gov.br/mre/pt-br/canais_atendiment...,17/04/2026,15h23,136,[O governo brasileiro saúda o anúncio de cessa...
9,Abertura de mercado para o Brasil no Vietnã - ...,https://www.gov.br/mre/pt-br/canais_atendiment...,17/04/2026,12h31,135,[O governo brasileiro concluiu negociações com...
10,Atos adotados por ocasião da Visita de Estado ...,https://www.gov.br/mre/pt-br/canais_atendiment...,17/04/2026,11h27,134,"[Foram adotados os seguintes atos, por ocasião..."


In [12]:
# saber quantidade de linhas e colunas do dataframe
df.shape

(30, 6)

In [13]:
# saber colunas disponiveis
df.columns

Index(['titulo', 'link', 'data', 'hora', 'num_nota', 'paragrafo'], dtype='object')

In [22]:
# selecionar uma coluna em especifico
df["titulo"]

,titulo
1,Aberturas de mercado para o Brasil no Togo - N...
2,Atos adotados por ocasião da Visita de Estado ...
3,Declaração Conjunta Brasil-Alemanha sobre apoi...
4,III Consultas Intergovernamentais de Alto Níve...
5,4ª Reunião de Alto Nível do Fórum Democracia S...
6,Declaração Conjunta Sobre a Situação em Cuba
7,Visita Oficial do Senhor Presidente da Repúbli...
8,Cessar-fogo no Líbano
9,Abertura de mercado para o Brasil no Vietnã - ...
10,Atos adotados por ocasião da Visita de Estado ...


In [23]:

# delimitar colunas do dataframe
df_delimitado = df[["titulo", "data"]]
df_delimitado

,titulo,data
1,Aberturas de mercado para o Brasil no Togo - N...,22/04/2026
2,Atos adotados por ocasião da Visita de Estado ...,20/04/2026
3,Declaração Conjunta Brasil-Alemanha sobre apoi...,20/04/2026
4,III Consultas Intergovernamentais de Alto Níve...,20/04/2026
5,4ª Reunião de Alto Nível do Fórum Democracia S...,19/04/2026
6,Declaração Conjunta Sobre a Situação em Cuba,18/04/2026
7,Visita Oficial do Senhor Presidente da Repúbli...,18/04/2026
8,Cessar-fogo no Líbano,17/04/2026
9,Abertura de mercado para o Brasil no Vietnã - ...,17/04/2026
10,Atos adotados por ocasião da Visita de Estado ...,17/04/2026


In [30]:

# primeiras (head), ultimas (tail) e linhas aleatórias (sample)
df.head(10)

,titulo,link,data,hora,num_nota,paragrafo
1,Aberturas de mercado para o Brasil no Togo - N...,https://www.gov.br/mre/pt-br/canais_atendiment...,22/04/2026,19h20,143,[O governo brasileiro concluiu negociações que...
2,Atos adotados por ocasião da Visita de Estado ...,https://www.gov.br/mre/pt-br/canais_atendiment...,20/04/2026,13h13,142,[Foram adotados os seguintes atos por ocasião ...
3,Declaração Conjunta Brasil-Alemanha sobre apoi...,https://www.gov.br/mre/pt-br/canais_atendiment...,20/04/2026,11h12,141,[No contexto da reunião bilateral entre o Pres...
4,III Consultas Intergovernamentais de Alto Níve...,https://www.gov.br/mre/pt-br/canais_atendiment...,20/04/2026,10h26,140,[1. Sob a presidência do Chanceler Federal Fri...
5,4ª Reunião de Alto Nível do Fórum Democracia S...,https://www.gov.br/mre/pt-br/canais_atendiment...,19/04/2026,07h58,139,"[Nosotras y nosotros, los Jefes de Estado y de..."
6,Declaração Conjunta Sobre a Situação em Cuba,https://www.gov.br/mre/pt-br/canais_atendiment...,18/04/2026,16h44,138,[À luz da evolução da situação em Cuba e das c...
7,Visita Oficial do Senhor Presidente da Repúbli...,https://www.gov.br/mre/pt-br/canais_atendiment...,18/04/2026,11h33,137,[O senhor Presidente da República realizará vi...
8,Cessar-fogo no Líbano,https://www.gov.br/mre/pt-br/canais_atendiment...,17/04/2026,15h23,136,[O governo brasileiro saúda o anúncio de cessa...
9,Abertura de mercado para o Brasil no Vietnã - ...,https://www.gov.br/mre/pt-br/canais_atendiment...,17/04/2026,12h31,135,[O governo brasileiro concluiu negociações com...
10,Atos adotados por ocasião da Visita de Estado ...,https://www.gov.br/mre/pt-br/canais_atendiment...,17/04/2026,11h27,134,"[Foram adotados os seguintes atos, por ocasião..."


In [32]:
df.describe(include="all")

,titulo,link,data,hora,num_nota,paragrafo
count,30,30,30,30,30,30
unique,30,30,14,28,30,30
top,Aberturas de mercado para o Brasil no Togo - N...,https://www.gov.br/mre/pt-br/canais_atendiment...,17/04/2026,12h31,143,[O governo brasileiro concluiu negociações que...
freq,1,1,4,2,1,1


In [36]:
df.isnull().sum()

,0
titulo,0
link,0
data,0
hora,0
num_nota,0
paragrafo,0


In [38]:
# verificar linhas duplicadas
df.duplicated().sum()

TypeError: unhashable type: 'list'